# W8C1 Lab: How a Model Chooses the Next Word

Run every cell from the top. **Everything already works.**

Uses distilgpt2 (about 350 MB), downloaded once and then cached.

Today you will:

1. Generate text with a real GPT and change how it picks words.
2. See temperature, top-k and top-p reshape the same distribution.
3. Find the setting that makes a model repeat itself forever.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, logging

logging.set_verbosity_error()
torch.manual_seed(0)

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()

PROMPT = "The best thing about living in a small town is"
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("prompt:", PROMPT)

## Part 1. The model does not write text, it scores words

At every step the model produces one number for every word in its
vocabulary. Everything after that is YOUR choice about how to pick one.

In [ ]:
# GIVEN. The raw distribution over the next word.
ids = tokenizer(PROMPT, return_tensors="pt")
with torch.no_grad():
    logits = model(**ids).logits[0, -1]

probs = logits.softmax(dim=-1)
top = probs.topk(10)
words = [tokenizer.decode([i]).strip() or "(space)" for i in top.indices]

print(f"the model scored all {len(probs):,} words in its vocabulary.")
print("the ten it likes best:")
for w, p in zip(words, top.values):
    print(f"   {w:<14} {p:.3f}")

plt.figure(figsize=(7, 3))
plt.bar(words, top.values.numpy(), color="#7C2529")
plt.xticks(rotation=45, ha="right"); plt.ylabel("probability")
plt.title("What comes after the prompt?"); plt.tight_layout(); plt.show()

## Part 2. The three knobs, drawn

Temperature stretches or flattens the whole distribution. Top-k keeps the
k best. Top-p keeps just enough words to reach probability p. Watch each
one reshape the very same numbers.

In [ ]:
# GIVEN. The same distribution under four settings.
def reshape(logits, temperature=1.0, top_k=None, top_p=None):
    x = logits / temperature
    p = x.softmax(dim=-1)
    if top_k:
        keep = p.topk(top_k).indices
        mask = torch.zeros_like(p, dtype=torch.bool); mask[keep] = True
        p = torch.where(mask, p, torch.zeros_like(p))
    if top_p:
        order = p.argsort(descending=True)
        cumulative = p[order].cumsum(0)
        cut = (cumulative < top_p).sum() + 1
        mask = torch.zeros_like(p, dtype=torch.bool); mask[order[:cut]] = True
        p = torch.where(mask, p, torch.zeros_like(p))
    return p / p.sum()

settings = [("temperature 0.5", dict(temperature=0.5)),
            ("temperature 1.0", dict()),
            ("temperature 2.0", dict(temperature=2.0)),
            ("top-k = 5", dict(top_k=5))]

fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
for ax, (name, kw) in zip(axes, settings):
    p = reshape(logits, **kw)
    t = p.topk(8)
    ax.bar([tokenizer.decode([i]).strip() or "_" for i in t.indices],
           t.values.numpy(), color="#7C2529")
    ax.set_title(name, fontsize=9); ax.tick_params(axis="x", rotation=75, labelsize=6)
    alive = int((p > 1e-9).sum())
    ax.set_xlabel(f"{alive:,} words possible", fontsize=7)
plt.tight_layout(); plt.show()
print("Low temperature = confident and boring. High = adventurous and unreliable.")
print("top-k throws the tail away entirely, so nonsense words can never be picked.")

In [ ]:
# GIVEN. Generate real text, three ways.
def generate(**kw):
    torch.manual_seed(0)
    out = model.generate(**ids, max_new_tokens=30, pad_token_id=tokenizer.eos_token_id, **kw)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print("--- greedy (always the top word) ---")
print(generate(do_sample=False))
print()
print("--- sampling, temperature 1.0 ---")
print(generate(do_sample=True, temperature=1.0, top_k=0))
print()
print("--- sampling, top-p 0.9 ---")
print(generate(do_sample=True, top_p=0.9, top_k=0))

In [ ]:
# ================== YOUR TURN 1 ==================
# Greedy decoding is the one that repeats itself. Look at its output
# above, then try to make sampling do the same thing by turning the
# temperature right down.
#
# Try 0.1, then 0.7, then 1.5.
#
# Expected: at 0.1 sampling looks almost exactly like greedy, loops included,
#           because a low temperature makes the top word overwhelmingly likely.
#           At 1.5 the text stops making sense. Somewhere near 0.7 to 0.9 is where
#           most real systems sit.
# ===============================================
TEMPERATURE = 0.1          # <-- try 0.7, then 1.5

text = generate(do_sample=True, temperature=TEMPERATURE, top_k=0)
print(f"temperature {TEMPERATURE}")
print(text)

words = text.split()
repeats = len(words) - len(set(words))
print(f"\nrepeated words: {repeats} out of {len(words)}")

In [ ]:
# ================== YOUR TURN 2 ==================
# top-p (nucleus) sampling keeps only the words needed to reach a total
# probability of p. Small p means a safe, small pool.
#
# Print how many words survive at each setting.
#
# Expected: p = 0.5 usually leaves a handful of words. p = 0.99 leaves hundreds.
#           This is why top-p adapts: when the model is confident, few words reach
#           the threshold; when it is unsure, many do. A fixed top-k cannot do that.
# ===============================================
for p in (0.5, 0.9, 0.99):
    reshaped = reshape(logits, top_p=p)
    alive = int((reshaped > 1e-9).sum())
    print(f"top-p {p:<5} keeps {alive:>4} words")

MY_P = 0.9          # <-- change me
print()
print(f"--- generated with top-p {MY_P} ---")
print(generate(do_sample=True, top_p=MY_P, top_k=0))

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   0.1 reproduces greedy decoding almost exactly, repetition and all. Dividing
#   the scores by a small number exaggerates the gaps, so after softmax the top
#   word takes nearly all the probability. Greedy is just temperature -> 0.
#
# YOUR TURN 2
#   The pool size changes with the model's confidence, which is the whole point
#   of top-p. After a confident prompt, p = 0.9 might keep 3 words; after an
#   ambiguous one, several hundred. top-k = 50 keeps 50 either way, which is too
#   few when the model is unsure and too many when it is certain.